# Part 04 — Gradients & Backpropagation

Gradients are the **feedback signal** that tells a model which direction to adjust its parameters to reduce its mistakes.

```
Training loop (simplified):
  1. Forward pass  → compute output from current weights
  2. Loss          → measure how wrong the output is
  3. Backward pass → compute ∂loss/∂weight for every parameter  ← gradients
  4. Update        → weight -= learning_rate × gradient
  5. Repeat
```

### What this notebook covers
| Section | Key idea |
|---------|----------|
| Forward pass + loss | Compute output, measure error |
| Chain rule & backprop | How gradients flow backward through layers |
| Gradient descent | Using gradients to update weights |
| Training loop | Full convergence walkthrough with visualization |
| Learning rate | Too small, just right, too large — visual comparison |
| `torch.no_grad()` | Why and when to disable gradient tracking |
| Parameters & bias | The two learnable quantities in every layer |

---
## What is a Gradient?

A gradient is the **slope of the loss function** with respect to a parameter.

> Analogy: You're blindfolded on a hilly surface, trying to walk downhill. The gradient tells you: *"The ground slopes up to your left"* — so you step right. Repeat until you reach the valley (minimum loss).

### The chain rule in one line

For a neural network `loss = f(output) = f(g(w))`:

```
∂loss     ∂loss    ∂output
────── =  ────── × ───────    ← chain rule
 ∂w       ∂output    ∂w
```

**Concrete example** — model: `output = w × x + b`,  loss: `(output − target)²`

```
∂loss/∂output = 2 × (output − target)          ← derivative of squared error
∂output/∂w   = x                                ← derivative of linear layer

∂loss/∂w      = 2 × (output − target) × x      ← chain rule product

With: output=7, target=10, x=3:
  ∂loss/∂w  = 2 × (7 − 10) × 3 = 2 × (−3) × 3 = −18
  ∂loss/∂b  = 2 × (7 − 10) × 1 = −6
```

**What the sign tells you:**
- Gradient = −18 → loss *decreases* as w *increases* → increase w to reduce loss
- `new_w = old_w − lr × gradient = 2.0 − 0.1 × (−18) = 3.8`  ✓ moved toward 10

---
## Forward Pass + Backward Pass — PyTorch traces the math automatically

In [ ]:
import torch

# Initialize
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([3.0])
target = torch.tensor([10.0])
learning_rate = 0.01 # The learning rate needs to be small enough to avoid oscillation.

print("Training Progress:")
print("Iteration | Weight | Bias  | Output | Loss")
print("-" * 45)

for iteration in range(50):
    # Forward pass
    output = w * x + b
    loss = (output - target)**2

    print(f"{iteration:9d} | {w.item():6.2f} | {b.item():5.2f} | {output.item():6.2f} | {loss.item():6.2f}")

    # Backward pass
    if w.grad is not None:
        w.grad.zero_()
    if b.grad is not None:
        b.grad.zero_()

    loss.backward()

    # Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # Stop if close enough
    if abs(output.item() - target.item()) < 0.01:
        print(f"Converged at iteration {iteration}!")
        break

print(f"\nFinal: w={w.item():.3f}, b={b.item():.3f}")
print(f"Final output: {(w*x + b).item():.3f} (target: {target.item()})")

---
## Training Loop — Watching the Model Converge

---
## Visualizing Gradient Descent

The loss landscape for `loss = (w×3 + b − 10)²` is a bowl.  
Gradient descent rolls the parameter down the slope toward the minimum.

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <img src="https://raw.githubusercontent.com/sprashant433/GenAI/main/images/gradient_descent_viz.png" style="width:50%; height:auto;" />
</div>

```
Converged in 27 iterations
```

---
## Learning Rate — The Most Important Hyperparameter

`lr` controls how big each update step is.  
Too small → painfully slow. Too large → overshoots and diverges.

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <img src="https://raw.githubusercontent.com/sprashant433/GenAI/main/images/learning_rate_comparison.png" style="width:50%; height:auto;" />
</div>

```
Rule of thumb:
  Start at lr=1e-3, try 1e-4 and 1e-2 and pick the fastest stable one.
  Red flag: loss goes up instead of down → halve the learning rate.
```

---
## `torch.no_grad()` — Training vs Inference

During **training** PyTorch builds a computation graph of every operation so it can  
backpropagate. During **inference** that graph is wasted memory and CPU time.

| | `requires_grad=True` (default) | `torch.no_grad()` |
|--|-------------------------------|-------------------|
| Purpose | Training | Inference / evaluation |
| Computation graph | Built and stored | Not built |
| Memory | Higher (all intermediates kept) | Lower |
| Speed | Slower | ~30% faster |
| `loss.backward()` | Works | Raises `RuntimeError` |

**Rule:** wrap every evaluation loop in `with torch.no_grad():`

---
## Memory: Computation Graph vs No Graph

Every intermediate tensor in the forward pass must be kept in memory for backprop.  
For a 12-layer BERT, that means retaining activations for every layer.

```python
# Memory estimate
seq_len, d_model, n_layers = 512, 768, 12
bytes_per_activation = seq_len * d_model * 4   # float32
train_mem_mb = bytes_per_activation * (n_layers + 2) / 1e6
infer_mem_mb = bytes_per_activation / 1e6
print(f"Activation memory (BERT, seq_len={seq_len}):")
print(f"  Training (all layers): ~{train_mem_mb:.1f} MB  (×{n_layers+2} layers)")
print(f"  Inference (output only): ~{infer_mem_mb:.1f} MB")
print(f"  Ratio: ~{train_mem_mb/infer_mem_mb:.0f}× more memory during training")
```

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <img src="https://raw.githubusercontent.com/sprashant433/GenAI/main/images/gradient_memory.png" style="width:50%; height:auto;" />
</div>

```
Activation memory (BERT, seq_len=512):
  Training (all layers): ~22.0 MB  (×14 layers)
  Inference (output only): ~1.6 MB
  Ratio: ~14× more memory during training
```

---
## Parameters & Bias — The Two Learnable Quantities

Every linear transformation in a neural network has exactly two types of learnable parameters:

```
output = input @ W + b

  W  (weight matrix) — scales and rotates the input
  b  (bias vector)   — shifts the output regardless of input
```

**Why bias matters:**
- Without bias: if `input = 0`, output is always `0` — the neuron can't fire independently
- With bias: the neuron has a "default activation level" it can adjust

```python
# In BERT's first attention layer (query projection):
W shape: [768, 768]  →  768 × 768 = 589,824 parameters
b shape: [768]       →  768 parameters
Total: 590,592 parameters in just this one sub-layer

# All parameters in a transformer block (BERT-base):
#   Attention Q/K/V + output:  4 × (768×768 + 768)    = 2,362,368
#   MLP FC1 + FC2:             (768×3072 + 3072) + (3072×768 + 768) = 4,722,432
#   LayerNorm ×2:              2 × (768 + 768)         = 3,072
#   ──────────────────────────────────────────────────────────────
#   Per block total:           ≈ 7,087,872
#   × 12 layers:               ≈ 85 million parameters
```

**During training**, gradients flow back to every `W` and every `b`.  
PyTorch's autograd engine computes `∂loss/∂W` and `∂loss/∂b` automatically via the chain rule.

---
## Summary

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| **Gradient** | `∂loss/∂param` — slope of loss w.r.t. parameter | Tells the optimizer which direction to move |
| **Backpropagation** | Chain rule applied layer by layer backward | Efficiently computes all gradients in one pass |
| **Gradient descent** | `param -= lr × gradient` | The update rule that makes models learn |
| **Learning rate** | Step size for each update | Too large → diverge; too small → slow |
| **Computation graph** | PyTorch's record of every operation | Required for backprop; skip it for inference |
| **`torch.no_grad()`** | Disables graph building | ~30% speedup + lower memory at inference time |
| **Weight (W)** | Matrix that transforms activations | Learned pattern detector |
| **Bias (b)** | Vector added to output | Shifts when a neuron fires |

```python
# The 4-line gradient descent loop — memorize this pattern:
output = model(x)               # 1. forward
loss   = criterion(output, y)   # 2. loss
optimizer.zero_grad()           # 3. clear old gradients
loss.backward()                 # 4. compute new gradients
optimizer.step()                # 5. update parameters
```